# 🌱 CSIRO Image2Biomass - Simplified Solution

**Minimal dependencies, maximum compatibility**

⚙️ **Notebook Settings**:
- Accelerator: **GPU T4 x2**
- Internet: **ON** (will turn OFF before submission)
- Persistence: Files only

## 📥 Step 1: Install ONLY Essential Packages

In [ ]:
%%time
# Install only timm - everything else is pre-installed in Kaggle
!pip install -q timm==0.9.12

import torch
import torchvision
print(f"✅ Setup complete!")
print(f"PyTorch: {torch.__version__}")
print(f"Torchvision: {torchvision.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 🔧 Step 2: Define Dataset (Using torchvision transforms)

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import random

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)

class BiomassDataset(Dataset):
    def __init__(self, df, image_dir, img_size=384, transform=None, is_train=True):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.img_size = img_size
        self.transform = transform
        self.is_train = is_train
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Load image
        img_name = row.get('image_path', f"{row['id']}.jpg")
        img_path = os.path.join(self.image_dir, img_name)
        
        try:
            image = Image.open(img_path).convert('RGB')
        except:
            # Fallback: create a dummy image if file not found
            image = Image.new('RGB', (self.img_size, self.img_size), (0, 0, 0))
        
        if self.transform:
            image = self.transform(image)
        
        sample = {'image': image, 'id': row.get('id', idx)}
        
        if self.is_train:
            target_col = 'target' if 'target' in row else 'biomass'
            if target_col in row:
                sample['target'] = torch.tensor([row[target_col]], dtype=torch.float32)
        
        return sample

# Define transforms using torchvision (more stable than albumentations)
def get_train_transforms(img_size):
    return transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomRotation(30),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

def get_valid_transforms(img_size):
    return transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

print("✅ Dataset defined")

## 🤖 Step 3: Define Model

In [ ]:
import timm

class BiomassModel(nn.Module):
    def __init__(self, model_name='efficientnet_b3', pretrained=True, dropout=0.3):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=pretrained, 
                                         num_classes=0, global_pool='avg')
        
        # Get feature dimension
        with torch.no_grad():
            dummy = torch.randn(1, 3, 224, 224)
            features = self.backbone(dummy)
            feat_dim = features.shape[1]
        
        self.head = nn.Sequential(
            nn.Linear(feat_dim, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 1)
        )
    
    def forward(self, x):
        if isinstance(x, dict):
            x = x['image']
        features = self.backbone(x)
        output = self.head(features)
        return output

print("✅ Model defined")

## 📊 Step 4: Load Data

In [ ]:
# Configuration
CONFIG = {
    'data_dir': '/kaggle/input/csiro-biomass',
    'img_size': 384,
    'batch_size': 16,
    'num_epochs': 15,
    'lr': 3e-4,
    'n_folds': 5,
    'seed': 42,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu'
}

print(f"Using device: {CONFIG['device']}\n")

# Load train data
train_df = pd.read_csv(f"{CONFIG['data_dir']}/train.csv")
print(f"Train samples: {len(train_df)}")
print(f"Columns: {list(train_df.columns)}")
print("\nFirst few rows:")
print(train_df.head())

## 🏋️ Step 5: Training Functions

In [ ]:
from tqdm.auto import tqdm

class AverageMeter:
    def __init__(self):
        self.reset()
    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0
    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

def train_one_epoch(model, loader, criterion, optimizer, device, scaler):
    model.train()
    losses = AverageMeter()
    
    pbar = tqdm(loader, desc='Training')
    for batch in pbar:
        images = batch['image'].to(device)
        targets = batch['target'].to(device)
        
        with torch.cuda.amp.autocast():
            preds = model(images)
            loss = criterion(preds, targets)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()
        
        losses.update(loss.item(), images.size(0))
        pbar.set_postfix({'loss': f'{losses.avg:.4f}'})
    
    return losses.avg

def validate(model, loader, criterion, device):
    model.eval()
    losses = AverageMeter()
    preds_list = []
    targets_list = []
    
    with torch.no_grad():
        for batch in tqdm(loader, desc='Validation'):
            images = batch['image'].to(device)
            targets = batch['target'].to(device)
            
            preds = model(images)
            loss = criterion(preds, targets)
            
            losses.update(loss.item(), images.size(0))
            preds_list.append(preds.cpu().numpy())
            targets_list.append(targets.cpu().numpy())
    
    preds_list = np.concatenate(preds_list)
    targets_list = np.concatenate(targets_list)
    rmse = np.sqrt(np.mean((preds_list - targets_list)**2))
    
    return losses.avg, rmse

print("✅ Training functions defined")

## 🎯 Step 6: Train Model with Cross-Validation

In [ ]:
%%time
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

# Simple train/val split (no sklearn KFold to avoid scipy dependency)
# Manual k-fold implementation
def simple_kfold(df, n_splits=5, seed=42):
    np.random.seed(seed)
    indices = np.arange(len(df))
    np.random.shuffle(indices)
    
    fold_size = len(df) // n_splits
    
    for fold in range(n_splits):
        val_start = fold * fold_size
        val_end = (fold + 1) * fold_size if fold < n_splits - 1 else len(df)
        
        val_idx = indices[val_start:val_end]
        train_idx = np.concatenate([indices[:val_start], indices[val_end:]])
        
        yield fold, train_idx, val_idx

# Create checkpoint directory
os.makedirs('/kaggle/working/checkpoints', exist_ok=True)

fold_scores = []

for fold, train_idx, valid_idx in simple_kfold(train_df, CONFIG['n_folds'], CONFIG['seed']):
    print(f"\n{'='*60}")
    print(f"Fold {fold}")
    print(f"{'='*60}\n")
    
    # Create datasets
    train_data = train_df.iloc[train_idx].reset_index(drop=True)
    valid_data = train_df.iloc[valid_idx].reset_index(drop=True)
    
    train_dataset = BiomassDataset(
        train_data,
        f"{CONFIG['data_dir']}/images",
        CONFIG['img_size'],
        get_train_transforms(CONFIG['img_size']),
        is_train=True
    )
    
    valid_dataset = BiomassDataset(
        valid_data,
        f"{CONFIG['data_dir']}/images",
        CONFIG['img_size'],
        get_valid_transforms(CONFIG['img_size']),
        is_train=True
    )
    
    train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'],
                            shuffle=True, num_workers=2, pin_memory=True)
    valid_loader = DataLoader(valid_dataset, batch_size=CONFIG['batch_size'],
                            shuffle=False, num_workers=2, pin_memory=True)
    
    # Model
    model = BiomassModel('efficientnet_b3', pretrained=True, dropout=0.3)
    model = model.to(CONFIG['device'])
    
    # Optimizer & Scheduler
    optimizer = AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=CONFIG['num_epochs'])
    criterion = nn.SmoothL1Loss()
    scaler = torch.cuda.amp.GradScaler()
    
    best_rmse = float('inf')
    patience = 7
    patience_counter = 0
    
    # Training loop
    for epoch in range(CONFIG['num_epochs']):
        print(f"\nEpoch {epoch+1}/{CONFIG['num_epochs']}")
        
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, 
                                    CONFIG['device'], scaler)
        valid_loss, valid_rmse = validate(model, valid_loader, criterion, CONFIG['device'])
        
        scheduler.step()
        
        print(f"Train Loss: {train_loss:.4f}, Valid Loss: {valid_loss:.4f}, Valid RMSE: {valid_rmse:.4f}")
        
        # Save best model
        if valid_rmse < best_rmse:
            best_rmse = valid_rmse
            patience_counter = 0
            torch.save({
                'model': model.state_dict(),
                'rmse': valid_rmse,
                'epoch': epoch
            }, f'/kaggle/working/checkpoints/best_fold{fold}.pth')
            print(f"✅ Best model saved! RMSE: {valid_rmse:.4f}")
        else:
            patience_counter += 1
        
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break
    
    fold_scores.append(best_rmse)
    print(f"\nFold {fold} Best RMSE: {best_rmse:.4f}")

print(f"\n{'='*60}")
print(f"Mean CV RMSE: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}")
print(f"{'='*60}")

## ⚠️ IMPORTANT: Turn OFF Internet Before Next Steps

**Before running inference and creating submission:**
1. Go to Notebook Settings (right side)
2. Turn **Internet → OFF**
3. Save settings
4. Continue with cells below

## 🔮 Step 7: Generate Predictions

In [ ]:
# Load test data
test_df = pd.read_csv(f"{CONFIG['data_dir']}/test.csv")
print(f"Test samples: {len(test_df)}")

test_dataset = BiomassDataset(
    test_df,
    f"{CONFIG['data_dir']}/images",
    CONFIG['img_size'],
    get_valid_transforms(CONFIG['img_size']),
    is_train=False
)

test_loader = DataLoader(test_dataset, batch_size=32, 
                        shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
%%time
# Inference with ensemble
all_predictions = []

for fold in range(CONFIG['n_folds']):
    checkpoint_path = f"/kaggle/working/checkpoints/best_fold{fold}.pth"
    
    if not os.path.exists(checkpoint_path):
        print(f"Checkpoint for fold {fold} not found, skipping")
        continue
    
    print(f"Loading fold {fold}...")
    model = BiomassModel('efficientnet_b3', pretrained=False)
    checkpoint = torch.load(checkpoint_path)
    model.load_state_dict(checkpoint['model'])
    model = model.to(CONFIG['device'])
    model.eval()
    
    fold_preds = []
    with torch.no_grad():
        for batch in tqdm(test_loader, desc=f"Fold {fold}"):
            images = batch['image'].to(CONFIG['device'])
            preds = model(images)
            fold_preds.append(preds.cpu().numpy())
    
    fold_preds = np.concatenate(fold_preds)
    all_predictions.append(fold_preds)
    print(f"Fold {fold} RMSE: {checkpoint['rmse']:.4f}")

# Ensemble predictions
final_predictions = np.mean(all_predictions, axis=0).squeeze()
print(f"\n✅ Generated {len(final_predictions)} predictions")

## 📊 Step 8: Create Submission

In [ ]:
# Create submission
submission = pd.DataFrame({
    'id': test_df['id'],
    'biomass': final_predictions
})

submission.to_csv('/kaggle/working/submission.csv', index=False)

print("✅ Submission created!\n")
print("Preview:")
print(submission.head(10))
print(f"\nStatistics:")
print(submission['biomass'].describe())
print(f"\nSaved to: /kaggle/working/submission.csv")
print("\n📥 Download from Output tab or submit directly!")

## ✅ Done!

**Next steps:**
1. ✅ Internet is already OFF (if you turned it off earlier)
2. Download `submission.csv` from Output tab
3. Go to competition page and submit!

**Or submit directly from notebook:**
- If internet is OFF, you can use Kaggle's submission system